In [ ]:

import tensorflow as tf

from tensorflow.keras import datasets, layers, models
import matplotlib.pyplot as plt

In [ ]:
import os
print(os.listdir("/kaggle/input"))

In [ ]:
import os
import tensorflow as tf

# 1. Automatically find the correct train and test paths on Kaggle
base_input_dir = "/kaggle/input"
train_path = None
test_path = None

for root, dirs, files in os.walk(base_input_dir):
    if root.endswith("train") and not train_path:
        train_path = root
    elif root.endswith("test") and not test_path:
        test_path = root

print(f" Detected Train Path: {train_path}")
print(f" Detected Test Path: {test_path}")

# 2. Safety check to make sure paths were found
if not train_path or not test_path:
    raise FileNotFoundError(
        "Could not automatically locate 'train' or 'test' directories. "
        f"Please verify your dataset is added. Current contents: {os.listdir(base_input_dir)}"
    )

train_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="training",
    seed=42,
    image_size=(224,224),
    batch_size=32
)

val_ds = tf.keras.utils.image_dataset_from_directory(
    train_path,
    validation_split=0.2,
    subset="validation",
    seed=42,
    image_size=(224,224),
    batch_size=32
)

test_ds = tf.keras.utils.image_dataset_from_directory(
    test_path,
    shuffle=False,
    image_size=(224,224),
    batch_size=32
)
# test_ds2 = tf.keras.utils.image_dataset_from_directory(
#     test_path2,
#     shuffle=False,
#     image_size=(224,224),
#     batch_size=32
# )

In [ ]:
base_model = tf.keras.applications.MobileNetV2(
    input_shape=(224, 224, 3),
    include_top=False, 
    weights='imagenet'
)
base_model.trainable = False # Freeze the base initially

model = models.Sequential([
    layers.Input(shape=(224, 224, 3)),
    layers.Lambda(tf.keras.applications.mobilenet_v2.preprocess_input),
    base_model,


    layers.GlobalAveragePooling2D(),
    layers.BatchNormalization(),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(4)
    
])

In [ ]:
model.compile(
    optimizer=tf.keras.optimizers.Adam(1e-3),
    loss=tf.keras.losses.SparseCategoricalCrossentropy(from_logits=True),
    metrics=["accuracy"]
)


history = model.fit(
    train_ds,
    validation_data=val_ds,
    epochs=10
)

In [ ]:
model.evaluate(test_ds)

In [ ]:
model.summary()

In [ ]:
history.history